In [0]:
%pip install --force-reinstall databricks-automl-runtime==0.2.20.14
%pip install mlflow==2.20.0
dbutils.library.restartPython()

# Prophet training
- This is an auto-generated notebook.
- To reproduce these results, attach this notebook to Serverless compute, and rerun it.
- Compare trials in the [MLflow experiment](#mlflow/experiments/1811583766169908).
- Clone this notebook into your project folder by selecting **File > Clone** in the notebook toolbar.

In [0]:
import mlflow
import databricks.automl_runtime

target_col = "avg_temp"
time_col = "date"
unit = "day"

frequency_quantity = 1


split_col = "split_col"


horizon = 365

## Load Data

In [0]:
import mlflow
import os
import uuid
import shutil
import pandas as pd
import pyspark.pandas as ps

# Create temp directory to download input data from MLflow
input_temp_dir = os.path.join(os.environ["SPARK_LOCAL_DIRS"], "tmp", str(uuid.uuid4())[:8])
os.makedirs(input_temp_dir)

# Download the artifact and read it into a pandas DataFrame
input_data_path = mlflow.artifacts.download_artifacts(run_id="24d9aa45d0a44656aac04830a88a8df7", artifact_path="data", dst_path=input_temp_dir)

input_file_path = os.path.join(input_data_path, "training_data")
df_loaded = ps.from_pandas(pd.read_parquet(input_file_path))

# Preview data
# display(df_loaded.head(5))

## Aggregate data by `time_col` and `split_col`
Group the data by `time_col` and `split_col`, and take average if there are multiple `target_col` values in the same group.

In [0]:
group_cols = [time_col]
group_cols = group_cols + [split_col]




df_aggregated = df_loaded \
  .groupby(group_cols) \
  .agg(y=(target_col, 'avg')) \
  .reset_index()

# display(df_aggregated.head(5))

## Train Prophet model
- Log relevant metrics to MLflow to track runs
- All the runs are logged under [this MLflow experiment](#mlflow/experiments/1811583766169908)
- Change the model parameters and re-run the training cell to log a different trial to the MLflow experiment

In [0]:
import logging

# disable informational messages from prophet
logging.getLogger("py4j").setLevel(logging.WARNING)

In [0]:
result_columns = ["model_json", "mse", "rmse", "mae", "mape", "mdape", "smape", "coverage"]

def prophet_training(history_pd):
  from hyperopt import hp
  from databricks.automl_runtime.forecast.prophet.forecast import ProphetHyperoptEstimator

  seasonality_mode = ["additive", "multiplicative"]
  search_space =  {
    "changepoint_prior_scale": hp.loguniform("changepoint_prior_scale", -6.9, -0.69),
    "seasonality_prior_scale": hp.loguniform("seasonality_prior_scale", -6.9, 2.3),
    "holidays_prior_scale": hp.loguniform("holidays_prior_scale", -6.9, 2.3),
    "seasonality_mode": hp.choice("seasonality_mode", seasonality_mode)
  }
  country_holidays="US"
  run_parallel = True
 
  val_df = history_pd[history_pd[split_col] == "validate"]
  split_cutoff = pd.Timestamp(val_df['ds'].max())

  hyperopt_estim = ProphetHyperoptEstimator(horizon=horizon, frequency_unit=unit, frequency_quantity=frequency_quantity, metric="smape",interval_width=0.8,
                   country_holidays=country_holidays, search_space=search_space, num_folds=20, max_eval=10, trial_timeout=7200,
                   split_cutoff=split_cutoff,
                   random_state=607099633, is_parallel=run_parallel)

  spark.conf.set("spark.databricks.mlflow.trackHyperopt.enabled", "false")

  results_pd = hyperopt_estim.fit(history_pd)

  spark.conf.set("spark.databricks.mlflow.trackHyperopt.enabled", "true")
 
  return results_pd[result_columns]

In [0]:
import mlflow
from databricks.automl_runtime.forecast.prophet.model import mlflow_prophet_log_model, ProphetModel

with mlflow.start_run(experiment_id="1811583766169908") as mlflow_run:
  mlflow.set_tag("estimator_name", "Prophet")
  mlflow.log_param("holiday_country", "US")
  mlflow.log_param("interval_width", 0.8)
  mlflow.log_param("random_state", 607099633)
  df_aggregated = df_aggregated.rename(columns={time_col: "ds"})

  forecast_results = prophet_training(df_aggregated.to_pandas())
    
  # Log the metrics to mlflow
  metric_name_map = {"mse": "mean_squared_error", "rmse": "root_mean_squared_error", "mae": "mean_absolute_error",
                     "mape": "mean_absolute_percentage_error", "mdape": "mdape", "smape": "smape", "coverage": "coverage"}
    
  split_prefix = "test_"
  
  
  avg_metrics = forecast_results[metric_name_map.keys()].rename(columns=metric_name_map).mean().to_frame(name="mean_metrics").reset_index()
  avg_metrics["index"] = split_prefix + avg_metrics["index"].astype(str)
  avg_metrics.set_index("index", inplace=True)
  mlflow.log_metrics(avg_metrics.to_dict()["mean_metrics"])
  
  # Create mlflow prophet model
  model_json = forecast_results["model_json"].to_list()[0]
  prophet_model = ProphetModel(model_json, horizon, unit, frequency_quantity, time_col)
  
# Generate sample input dataframe
  sample_input = df_loaded.head(1).to_pandas()
  sample_input[time_col] = pd.to_datetime(sample_input[time_col])
  sample_input.drop(columns=[target_col], inplace=True)
  sample_input.drop(columns=[split_col], inplace=True)

  mlflow_prophet_log_model(prophet_model, sample_input=sample_input)

In [0]:
avg_metrics
# Uncomment to check the result details. By default, we only show the averaged metric becuase
# displaying the forecast result can take up a lot of storage for large datasets.
# forecast_results.head()